<img src="../../images/bwHPC_Logo_cmyk.svg" width="200" /> <img src="../../images/HochschuleEsslingen_Logo_RGB_DE.png" width="200" /> <img src="../../images/Konstanz_Logo.svg" width="200" /> <img src="../../images/KIT_Logo.png" width="200" />

# Refactoring with torch.nn

The classifier in [notebook 02](02_A_Neural_Network_From_Scratch.ipynb) works, and
it is about fifty lines with several things that are easy to get wrong: zeroing
gradients, slicing batches by hand, remembering `no_grad()`.

PyTorch has a tool for each of those. This notebook applies them **one at a time**,
and after every step the model is retrained to show the result is unchanged. Nothing
gets better at recognising digits here — the code gets shorter, and the number of
ways to make a silent mistake goes down.

That is the whole point of the exercise: each tool arrives only after you have felt
the problem it solves.

---

## Contents

1. [Starting point](#setup)
2. [nn.functional](#functional)
3. [nn.Module](#module)
4. [nn.Linear](#linear)
5. [torch.optim](#optim)
6. [Dataset](#dataset)
7. [DataLoader](#dataloader)
8. [Adding validation](#validation)
9. [fit() and get_data()](#fit)

<a id="setup"></a>
## 1. Starting point

Each notebook in this chapter runs on its own, so we begin by rebuilding the state
[notebook 02](02_A_Neural_Network_From_Scratch.ipynb) ended in: the data, the
weights, the model, and the loss and accuracy functions.

Nothing here is new — it is the previous notebook, compressed.

In [1]:
import gzip
import pickle

import torch

with gzip.open("mnist.pkl.gz", "rb") as file:
    ((x_train, y_train), (x_valid, y_valid), _) = pickle.load(file, encoding="latin-1")

x_train, y_train, x_valid, y_valid = map(
    torch.tensor, (x_train, y_train, x_valid, y_valid)
)

n, c = x_train.shape
print("training images:", tuple(x_train.shape))

training images: (50000, 784)


In [2]:
import math

import numpy as np
import matplotlib.pyplot as plt

bs = 64          # batch size
lr = 0.5         # learning rate
epochs = 2

weights = torch.randn(784, 10) / math.sqrt(784)
weights.requires_grad_()
bias = torch.zeros(10, requires_grad=True)


def log_softmax(x):
    return x - x.exp().sum(-1).log().unsqueeze(-1)


def model(xb):
    return log_softmax(xb @ weights + bias)


def nll(input, target):
    return -input[range(target.shape[0]), target].mean()


def accuracy(out, yb):
    return (torch.argmax(out, dim=1) == yb).float().mean()


loss_func = nll

xb = x_train[0:bs]
yb = y_train[0:bs]
print("loss:", loss_func(model(xb), yb).item())

loss: 2.344205856323242


<a id="functional"></a>
## 2. nn.functional

The easiest win first. `torch.nn.functional` — imported as `F` by convention —
contains ready-made activation and loss functions, so `log_softmax` and `nll` can go.

**`F.cross_entropy` does both at once**: it applies the log-softmax *and* computes
the negative log-likelihood. So the model no longer needs its own activation, and
becomes a single matrix multiplication.

In [3]:
import torch.nn.functional as F

loss_func = F.cross_entropy


def model(xb):
    return xb @ weights + bias


print("loss:    ", loss_func(model(xb), yb).item())
print("accuracy:", accuracy(model(xb), yb).item())

loss:     2.344205856323242
accuracy: 0.109375


Same numbers as before — two hand-written functions gone.

<a id="module"></a>
## 3. nn.Module

The weights are still loose variables, which is why the training loop has to name
each one when it updates them. Forget one and nothing complains.

**`nn.Module`** is PyTorch's container for a model. Subclass it, register the
parameters with **`nn.Parameter`**, and put the forward pass in a method called
`forward`. In exchange it keeps track of the parameters for you, through
`.parameters()` and `.zero_grad()`.

In [4]:
from torch import nn


class Mnist_Logistic(nn.Module):
    def __init__(self):
        super().__init__()
        self.weights = nn.Parameter(torch.randn(784, 10) / math.sqrt(784))
        self.bias = nn.Parameter(torch.zeros(10))

    def forward(self, xb):
        return xb @ self.weights + self.bias


model = Mnist_Logistic()

`model` is now an object rather than a function, but it is still *called* like one:
PyTorch runs `forward` behind the scenes.

In [5]:
print("loss before training:", loss_func(model(xb), yb).item())

loss before training: 2.318669557571411


And the update loop stops naming parameters individually. Where notebook 02 had

```python
weights -= weights.grad * lr
bias    -= bias.grad * lr
weights.grad.zero_()
bias.grad.zero_()
```

it can now loop over whatever the model happens to contain — which keeps working
when the model grows.

In [6]:
def fit():
    for epoch in range(epochs):
        for i in range((n - 1) // bs + 1):
            start_i = i * bs
            end_i = start_i + bs
            xb = x_train[start_i:end_i]
            yb = y_train[start_i:end_i]

            loss = loss_func(model(xb), yb)
            loss.backward()

            with torch.no_grad():
                for p in model.parameters():
                    p -= p.grad * lr
                model.zero_grad()


fit()
print("loss after training:", loss_func(model(xb), yb).item())

loss after training: 0.2234250158071518


<a id="linear"></a>
## 4. nn.Linear

Defining a weight matrix and a bias, then writing `xb @ self.weights + self.bias`, is
such a common pattern that PyTorch has a layer for it. **`nn.Linear`** creates both
tensors, initialises them sensibly, and applies them.

In [7]:
class Mnist_Logistic(nn.Module):
    def __init__(self):
        super().__init__()
        self.lin = nn.Linear(784, 10)

    def forward(self, xb):
        return self.lin(xb)


model = Mnist_Logistic()
print("loss before training:", loss_func(model(xb), yb).item())

loss before training: 2.317735433578491


The `fit` function from the previous step is untouched — it only ever asked the model
for its parameters, and does not care where they came from.

In [8]:
fit()
print("loss after training: ", loss_func(model(xb), yb).item())

loss after training:  0.22767756879329681


<a id="optim"></a>
## 5. torch.optim

The update rule is still written out by hand. `torch.optim` has it, along with every
other optimizer you are likely to want — including the **Adam** used throughout the
TensorFlow notebooks. Here we use plain **stochastic gradient descent**.

The two lines

```python
with torch.no_grad():
    for p in model.parameters(): p -= p.grad * lr
    model.zero_grad()
```

become `opt.step()` and `opt.zero_grad()`.

In [9]:
from torch import optim


def get_model():
    model = Mnist_Logistic()
    return model, optim.SGD(model.parameters(), lr=lr)


model, opt = get_model()
print("loss before training:", loss_func(model(xb), yb).item())

loss before training: 2.2973899841308594


In [10]:
for epoch in range(epochs):
    for i in range((n - 1) // bs + 1):
        start_i = i * bs
        end_i = start_i + bs
        xb = x_train[start_i:end_i]
        yb = y_train[start_i:end_i]

        loss = loss_func(model(xb), yb)
        loss.backward()

        opt.step()
        opt.zero_grad()

print("loss after training: ", loss_func(model(xb), yb).item())

loss after training:  0.07990224659442902


<a id="dataset"></a>
## 6. Dataset

What is left of the ugliness is the batching: `x_train[start_i:end_i]` and
`y_train[start_i:end_i]`, two slices that have to stay in step with each other.

A **`Dataset`** in PyTorch is anything with a length and an index. **`TensorDataset`**
wraps tensors into one, so the images and their labels can be indexed together.

In [11]:
from torch.utils.data import TensorDataset

train_ds = TensorDataset(x_train, y_train)

print("length:", len(train_ds))
images, labels = train_ds[0:bs]
print("one slice gives both:", tuple(images.shape), tuple(labels.shape))

length: 50000
one slice gives both: (64, 784) (64,)


In [12]:
model, opt = get_model()

for epoch in range(epochs):
    for i in range((n - 1) // bs + 1):
        xb, yb = train_ds[i * bs: i * bs + bs]        # one index, not two

        loss = loss_func(model(xb), yb)
        loss.backward()
        opt.step()
        opt.zero_grad()

print("loss after training:", loss_func(model(xb), yb).item())

loss after training: 0.08138210326433182


<a id="dataloader"></a>
## 7. DataLoader

The index arithmetic is still there. **`DataLoader`** takes a `Dataset` and hands out
batches one after another, so the loop becomes an ordinary `for` over the data.

In [13]:
from torch.utils.data import DataLoader

train_dl = DataLoader(train_ds, batch_size=bs)

model, opt = get_model()

for epoch in range(epochs):
    for xb, yb in train_dl:                          # no arithmetic at all
        loss = loss_func(model(xb), yb)
        loss.backward()
        opt.step()
        opt.zero_grad()

print("loss after training:", loss_func(model(xb), yb).item())

loss after training: 0.08127404749393463


Compare that inner loop with the one in notebook 02. Same computation, none of the
bookkeeping.

<a id="validation"></a>
## 8. Adding validation

Everything so far has been measured on the training data, which says nothing about
overfitting. The dataset came with a validation set, so we use it.

Two details worth knowing:

- the **training** data is shuffled, so that batches do not stay correlated from one
  epoch to the next. The validation data is not — shuffling costs time and cannot
  change the result
- the validation batches are **twice as large**. No gradients are stored for them, so
  they need less memory

`model.train()` and `model.eval()` switch the model between the two modes. It makes
no difference to this model, but layers like dropout and batch-norm behave differently
in each, so it is a habit worth having.

In [14]:
train_ds = TensorDataset(x_train, y_train)
train_dl = DataLoader(train_ds, batch_size=bs, shuffle=True)

valid_ds = TensorDataset(x_valid, y_valid)
valid_dl = DataLoader(valid_ds, batch_size=bs * 2)

In [15]:
model, opt = get_model()

for epoch in range(epochs):
    model.train()
    for xb, yb in train_dl:
        loss = loss_func(model(xb), yb)
        loss.backward()
        opt.step()
        opt.zero_grad()

    model.eval()
    with torch.no_grad():
        valid_loss = sum(loss_func(model(xb), yb) for xb, yb in valid_dl)

    print(f"epoch {epoch}   validation loss {valid_loss / len(valid_dl):.5f}")

epoch 0   validation loss 0.29375
epoch 1   validation loss 0.28754


<a id="fit"></a>
## 9. fit() and get_data()

One last tidy-up, this time ours rather than PyTorch's.

The loss is computed twice in almost the same way — once with an optimizer for
training, once without for validation. `loss_batch` handles both: pass an optimizer
and it takes a step, leave it out and it only measures.

In [16]:
def loss_batch(model, loss_func, xb, yb, opt=None):
    loss = loss_func(model(xb), yb)

    if opt is not None:
        loss.backward()
        opt.step()
        opt.zero_grad()

    return loss.item(), len(xb)

`fit` then runs the whole training, and `get_data` builds the two loaders.

In [17]:
def fit(epochs, model, loss_func, opt, train_dl, valid_dl):
    for epoch in range(epochs):
        model.train()
        for xb, yb in train_dl:
            loss_batch(model, loss_func, xb, yb, opt)

        model.eval()
        with torch.no_grad():
            losses, nums = zip(
                *[loss_batch(model, loss_func, xb, yb) for xb, yb in valid_dl]
            )
        val_loss = np.sum(np.multiply(losses, nums)) / np.sum(nums)

        print(f"epoch {epoch}   validation loss {val_loss:.5f}")


def get_data(train_ds, valid_ds, bs):
    return (
        DataLoader(train_ds, batch_size=bs, shuffle=True),
        DataLoader(valid_ds, batch_size=bs * 2),
    )

And now the entire process — data, model, training — is three lines.

In [18]:
train_dl, valid_dl = get_data(train_ds, valid_ds, bs)
model, opt = get_model()
fit(epochs, model, loss_func, opt, train_dl, valid_dl)

epoch 0   validation loss 0.31141
epoch 1   validation loss 0.28434


Those three lines do exactly what the fifty in notebook 02 did.

They are also **model-independent**. Nothing in `fit` assumes anything about what
`model` is — which is what [notebook 04](04_Convolutional_Neural_Networks.ipynb)
exploits, by swapping in a convolutional network and changing nothing else.